# EWSNet — Replication of Four Dataset Experiments

Reproduces the four benchmark experiments from:
> *Wavelet transform and edge loss-based three-stage segmentation model for retinal vessel*  
> Li et al., Biomedical Signal Processing and Control, 2023

**Datasets**: CHASE_DB1 · STARE · HRF · UWF

**Expected Google Drive layout** (all datasets under one root folder):
```
MyDrive/EWSNet_data/
├── chasedb1/
│   ├── train/  {img/, thin_mask/, thick_mask/, mask/}
│   └── test/   {img/, noise/, thin_mask/, thick_mask/, mask/}
├── stare/  (same structure)
├── hrf/    (same structure)
└── uwf/    (same structure)
```

**Thick/thin mask generation**: Use `tools/helpfunc.py::extract()` on each dataset's `mask/` folder to auto-generate `thick_mask/` and `thin_mask/` via morphological open/close.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Prepare Datasets (One-time Setup)

Reorganises raw Drive datasets into EWSNet-compatible structure and auto-generates `thick_mask/` + `thin_mask/`.

**Source** (your existing Drive layout):
```
MyDrive/dataset/
├── CHASEDB1/          ← flat folder: Image_01L.jpg + Image_01L_1stHO.png
├── STARE/             ← images/ | snd_label_vk/ | 1st_labels_ah/ | mask/
└── HRF/               ← healthy/ | glaucoma/ | all_2/  (each: .jpg + .tif mixed)
```

**Target** (created by this script):
```
MyDrive/SegModels/EWSNet/
├── chasedb1/train|test/{img, mask, thick_mask, thin_mask, noise}
├── stare/   train|test/{img, mask, thick_mask, thin_mask, noise}
└── hrf/     train|test/{img, mask, thick_mask, thin_mask, noise}
```

> **Run once** — skip this section on subsequent training runs.

In [ ]:
import os, cv2, shutil, random
import numpy as np
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
SRC_ROOT  = '/content/drive/MyDrive/dataset'          # your existing datasets
DST_ROOT  = '/content/drive/MyDrive/SegModels/EWSNet' # EWSNet training root
NOISE_STD = 10   # Gaussian noise std for test/noise/ folder

random.seed(42)

# ── Shared helpers ────────────────────────────────────────────────────────────

def make_ews_dirs(name):
    """Create EWSNet folder structure for one dataset."""
    for split in ['train', 'test']:
        for sub in ['img', 'mask', 'thick_mask', 'thin_mask']:
            os.makedirs(f'{DST_ROOT}/{name}/{split}/{sub}', exist_ok=True)
    os.makedirs(f'{DST_ROOT}/{name}/test/noise', exist_ok=True)


def split_thick_thin(mask_src, thick_dst, thin_dst):
    """Morphological open/close → thick and thin vessel masks (same as helpfunc.extract)."""
    img = cv2.imread(str(mask_src), cv2.IMREAD_GRAYSCALE)
    if img is None:                          # .ppm / .tif fallback
        tmp = cv2.imread(str(mask_src), cv2.IMREAD_COLOR)
        if tmp is not None:
            img = cv2.cvtColor(tmp, cv2.COLOR_BGR2GRAY)
    if img is None:
        print(f'  [WARN] cannot read {mask_src}')
        return
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    opened = cv2.morphologyEx(
        cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel), cv2.MORPH_OPEN, kernel)
    cv2.imwrite(str(thick_dst), cv2.bitwise_and(img, opened))
    cv2.imwrite(str(thin_dst),  cv2.bitwise_and(img, cv2.bitwise_not(opened)))


def save_noisy(src, dst):
    """Copy image with added Gaussian noise into noise/ folder."""
    img = cv2.imread(str(src), cv2.IMREAD_COLOR)
    if img is None:
        shutil.copy2(src, dst)
        return
    noise = np.random.normal(0, NOISE_STD, img.shape)
    cv2.imwrite(str(dst),
                np.clip(img.astype(np.int32) + noise.astype(np.int32), 0, 255).astype(np.uint8))


# ── CHASEDB1 ──────────────────────────────────────────────────────────────────
# Source: flat folder — Image_01L.jpg  |  Image_01L_1stHO.png
# EWSNet mask name pattern: {stem}_1stHO.png
# Split: sorted alphabetically — first 20 → train, last 8 → test

def process_chasedb1():
    src  = Path(SRC_ROOT) / 'CHASEDB1'
    name = 'chasedb1'
    make_ews_dirs(name)

    imgs = sorted(f for f in os.listdir(src) if f.lower().endswith('.jpg'))
    assert len(imgs) > 0, f'No .jpg images found in {src}'

    splits = {'train': imgs[:20], 'test': imgs[20:]}
    for split, img_list in splits.items():
        for img_name in img_list:
            stem      = Path(img_name).stem          # Image_01L
            mask_name = f'{stem}_1stHO.png'
            base      = Path(DST_ROOT) / name / split

            shutil.copy2(src / img_name,  base / 'img'  / img_name)
            shutil.copy2(src / mask_name, base / 'mask' / mask_name)
            split_thick_thin(
                base / 'mask' / mask_name,
                base / 'thick_mask' / mask_name,
                base / 'thin_mask'  / mask_name,
            )
            if split == 'test':
                save_noisy(base / 'img' / img_name, base / 'noise' / img_name)

    print(f'  CHASEDB1  — train: {len(splits["train"])}  test: {len(splits["test"])}')


# ── STARE ─────────────────────────────────────────────────────────────────────
# Source:
#   images/       — original images (*.ppm)
#   snd_label_vk/ — vessel annotations (*.vk.ppm)  ← EWSNet uses this
# EWSNet mask name pattern: {stem}.vk.ppm  (stored in mask/)
# thick/thin stored as {stem}.png
# Split: sorted — first 10 → train, last 10 → test

def process_stare():
    src       = Path(SRC_ROOT) / 'STARE'
    name      = 'stare'
    images_dir = src / 'images'
    vk_dir     = src / 'snd_label_vk'
    make_ews_dirs(name)

    imgs = sorted(f for f in os.listdir(images_dir) if not f.startswith('.'))
    assert len(imgs) > 0, f'No images found in {images_dir}'

    splits = {'train': imgs[:10], 'test': imgs[10:]}
    for split, img_list in splits.items():
        for img_name in img_list:
            stem      = Path(img_name).stem          # im0001
            mask_name = f'{stem}.vk.ppm'
            base      = Path(DST_ROOT) / name / split

            shutil.copy2(images_dir / img_name, base / 'img'  / img_name)
            shutil.copy2(vk_dir     / mask_name, base / 'mask' / mask_name)
            split_thick_thin(
                base / 'mask' / mask_name,
                base / 'thick_mask' / f'{stem}.png',
                base / 'thin_mask'  / f'{stem}.png',
            )
            if split == 'test':
                save_noisy(base / 'img' / img_name, base / 'noise' / img_name)

    print(f'  STARE     — train: {len(splits["train"])}  test: {len(splits["test"])}')


# ── HRF ───────────────────────────────────────────────────────────────────────
# Source: healthy/ | glaucoma/ | all_2/
#   Each subfolder contains .jpg images and .tif manual segmentations (same stem)
# EWSNet mask name pattern: {stem}.tif  (stored in mask/)
# thick/thin stored as {stem}.png
# Split: per-category 2/3 train, 1/3 test

def process_hrf():
    src  = Path(SRC_ROOT) / 'HRF'
    name = 'hrf'
    make_ews_dirs(name)

    categories = ['healthy', 'glaucoma', 'all_2']
    train_items, test_items = [], []

    for cat in categories:
        cat_dir = src / cat
        if not cat_dir.exists():
            print(f'  [WARN] {cat_dir} not found — skipping')
            continue
        imgs = sorted(f for f in os.listdir(cat_dir)
                      if f.lower().endswith('.jpg') and not f.startswith('.'))
        n_train = max(1, round(len(imgs) * 2 / 3))
        train_items.extend((cat, f) for f in imgs[:n_train])
        test_items.extend( (cat, f) for f in imgs[n_train:])
        print(f'    {cat}: {n_train} train / {len(imgs)-n_train} test')

    for split, items in [('train', train_items), ('test', test_items)]:
        for cat, img_name in items:
            stem      = Path(img_name).stem          # e.g. 01_h
            mask_name = f'{stem}.tif'
            cat_dir   = src / cat
            base      = Path(DST_ROOT) / name / split

            shutil.copy2(cat_dir / img_name,  base / 'img'  / img_name)
            shutil.copy2(cat_dir / mask_name, base / 'mask' / mask_name)
            split_thick_thin(
                base / 'mask' / mask_name,
                base / 'thick_mask' / f'{stem}.png',
                base / 'thin_mask'  / f'{stem}.png',
            )
            if split == 'test':
                save_noisy(base / 'img' / img_name, base / 'noise' / img_name)

    print(f'  HRF       — train: {len(train_items)}  test: {len(test_items)}')


# ── Run all ───────────────────────────────────────────────────────────────────
print('Organising datasets → ', DST_ROOT)
print()
process_chasedb1()
process_stare()
process_hrf()
print()
print('Done. Ready for training.')

## 2. Clone Repository and Install Dependencies

In [ ]:
!git clone https://github.com/xuecheng990531/EWSNet.git
%cd EWSNet

# Colab already provides: numpy, torch, torchvision, matplotlib, opencv, Pillow, tqdm
# Only install packages that are NOT pre-installed in Colab
!pip install albumentations==1.3.1 torchmetrics==1.0.1 PyWavelets==1.4.1 -q

## 3. Experiment Configuration

Edit `DRIVE_ROOT` to match your Google Drive path.  
Set `RUN_EXPERIMENTS[name] = False` to skip a dataset.

In [ ]:
import os

# ── Data root (matches DST_ROOT in the preprocessing script) ─────────────────
DRIVE_ROOT = '/content/drive/MyDrive/SegModels/EWSNet'
# ─────────────────────────────────────────────────────────────────────────────

EXPERIMENTS = {
    'chasedb1': {
        'train_dir':  f'{DRIVE_ROOT}/chasedb1/train',
        'test_dir':   f'{DRIVE_ROOT}/chasedb1/test',
        'epochs':     140,
        'batch_size': 2,
    },
    'stare': {
        'train_dir':  f'{DRIVE_ROOT}/stare/train',
        'test_dir':   f'{DRIVE_ROOT}/stare/test',
        'epochs':     140,
        'batch_size': 2,
    },
    'hrf': {
        'train_dir':  f'{DRIVE_ROOT}/hrf/train',
        'test_dir':   f'{DRIVE_ROOT}/hrf/test',
        'epochs':     140,
        'batch_size': 2,
    },
}

RUN_EXPERIMENTS = {
    'chasedb1': True,
    'stare':    True,
    'hrf':      True,
}

RESULTS_DRIVE_DIR = f'{DRIVE_ROOT}/results'
os.makedirs(RESULTS_DRIVE_DIR, exist_ok=True)

print('Experiments configured:')
for name, cfg in EXPERIMENTS.items():
    tag = 'RUN ' if RUN_EXPERIMENTS[name] else 'SKIP'
    print(f'  [{tag}] {name}  epochs={cfg["epochs"]}  bs={cfg["batch_size"]}')

## 4. Pre-flight: Verify Dataset Directories

In [ ]:
all_ok = True
for name, cfg in EXPERIMENTS.items():
    if not RUN_EXPERIMENTS[name]:
        continue
    for label, path in [('train', cfg['train_dir']), ('test', cfg['test_dir'])]:
        ok = os.path.exists(path)
        print(f'  [{"OK" if ok else "MISSING"}] {name}/{label}: {path}')
        if not ok:
            all_ok = False

if all_ok:
    print('\nAll directories found — ready to train.')
else:
    print('\nFix missing paths before running training.')

## 5. Define Training Function

Inline replication of `main.py::train()` with two bug-fixes from the original:
- `visual_output/thick|thin|all/` sub-directories are now created automatically
- `save_checkpoint` model/path order corrected (`thick` ↔ `refine` were swapped in `helpfunc.py`)

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchmetrics.classification import (
    BinaryF1Score, BinaryRecall, BinarySpecificity, BinaryAccuracy, BinaryAUROC
)
from torchvision.utils import save_image
from tqdm import tqdm

from models.thick_net import ThickNet
from models.resunet import ResUnet
from models.refine import denosing_module
from tools.dl import fundus_data
from tools.loss import Focal_IoU, EdgeLoss_BCE


def train_one_experiment(dataset_name, train_dir, test_dir, epochs, batch_size, device):
    """Full training + validation loop for one dataset. Returns best metrics dict."""

    # ── Directories ──────────────────────────────────────────────────────────
    ckpt_dir = f'ckpts/{dataset_name}'
    vis_dirs = [
        f'visual_output/{dataset_name}/thick',
        f'visual_output/{dataset_name}/thin',
        f'visual_output/{dataset_name}/all',
    ]
    for d in [ckpt_dir] + vis_dirs:
        os.makedirs(d, exist_ok=True)

    # ── Models ───────────────────────────────────────────────────────────────
    thinmodel  = ResUnet(channel=2, device=device).to(device)         # ThinNet
    thickmodel = ThickNet(in_channels=1).to(device)                   # ThickNet
    refine     = denosing_module(device=device, inchannel=1).to(device)  # RefineNet

    # ── Optimizers (ThinNet uses much smaller lr — original paper setting) ───
    opt_thin  = torch.optim.Adam(thinmodel.parameters(),  lr=1e-6, betas=(0.9, 0.999))
    opt_thick = torch.optim.Adam(thickmodel.parameters(), lr=1e-4, betas=(0.9, 0.999))
    opt_ref   = torch.optim.Adam(refine.parameters(),     lr=1e-4, betas=(0.9, 0.999))

    # ── Loss functions ───────────────────────────────────────────────────────
    #   ThickNet : BCE
    #   ThinNet  : Focal + IoU (theta=0.5)
    #   RefineNet: BCE + 4-direction Sobel edge loss (alpha=0.5)
    crit_thin  = Focal_IoU(theta=0.5).to(device)
    crit_thick = nn.BCELoss().to(device)
    crit_ref   = EdgeLoss_BCE(device=device, alpha=0.5).to(device)

    # ── Data ─────────────────────────────────────────────────────────────────
    train_ds = fundus_data(train_dir, mode='train', name=dataset_name, isnoise=False)
    test_ds  = fundus_data(test_dir,  mode='test',  name=dataset_name, isnoise=True)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=1,          shuffle=False, num_workers=0)

    writer    = SummaryWriter(ckpt_dir)
    best_f1   = 0.0
    best_metrics = {}

    for epoch in range(epochs + 1):

        # ── Train ─────────────────────────────────────────────────────────────
        thinmodel.train(); thickmodel.train(); refine.train()
        train_loss = 0.0
        pbar = tqdm(train_loader, colour='#5181D5',
                    desc=f'[{dataset_name}] Epoch {epoch}/{epochs}', dynamic_ncols=True)

        for img, thin_lbl, thick_lbl, all_lbl in pbar:
            img, thin_lbl, thick_lbl, all_lbl = (
                img.to(device), thin_lbl.to(device),
                thick_lbl.to(device), all_lbl.to(device)
            )

            # Stage 1: ThickNet
            thick_pred = thickmodel(img)
            loss_thick = crit_thick(thick_pred, thick_lbl)

            # Stage 2: ThinNet — conditioned on ThickNet output
            thin_pred = thinmodel(torch.cat([img, thick_pred], dim=1))
            loss_thin = crit_thin(thin_pred, thin_lbl)

            # Stage 3: RefineNet — fuse coarse predictions
            first_pred  = thick_pred + thin_pred
            second_pred = refine(first_pred)
            loss_ref    = crit_ref(second_pred, all_lbl)

            total_loss = loss_thin + loss_thick + loss_ref

            opt_thin.zero_grad(); opt_thick.zero_grad(); opt_ref.zero_grad()
            total_loss.backward()
            opt_thin.step(); opt_thick.step(); opt_ref.step()

            train_loss += total_loss.item() / len(train_loader)
            pbar.set_postfix({'loss': f'{train_loss:.5f}'})

        # ── Validate ──────────────────────────────────────────────────────────
        thinmodel.eval(); thickmodel.eval(); refine.eval()

        f1_m   = BinaryF1Score().to(device)
        acc_m  = BinaryAccuracy().to(device)
        sen_m  = BinaryRecall().to(device)
        spe_m  = BinarySpecificity().to(device)
        auc_m  = BinaryAUROC().to(device)

        with torch.no_grad():
            for idx, (img, thin_lbl, thick_lbl, all_lbl) in enumerate(test_loader):
                img, thick_lbl, thin_lbl, all_lbl = (
                    img.to(device), thick_lbl.to(device),
                    thin_lbl.to(device), all_lbl.to(device)
                )

                thick_val = thickmodel(img)
                thin_val  = thinmodel(torch.cat([img, thick_val], dim=1))
                pred_val  = refine(thick_val + thin_val)

                # Save per-dataset visual outputs (avoids cross-run overwrite)
                save_image(thick_val, f'visual_output/{dataset_name}/thick/thick_{idx+1}.png')
                save_image(thin_val,  f'visual_output/{dataset_name}/thin/thin_{idx+1}.png')
                save_image(pred_val,  f'visual_output/{dataset_name}/all/all_{idx+1}.png')

                pred_bin = (pred_val > 0.5).float()

            # Metrics on last test batch (consistent with original main.py)
            f1_val  = f1_m(pred_bin,  all_lbl).item()
            acc_val = acc_m(pred_bin,  all_lbl).item()
            sen_val = sen_m(pred_bin,  all_lbl).item()
            spe_val = spe_m(pred_bin,  all_lbl).item()
            auc_val = auc_m(pred_val,  all_lbl).item()

        writer.add_scalar('F1',  f1_val,  epoch)
        writer.add_scalar('Acc', acc_val, epoch)
        writer.add_scalar('Sen', sen_val, epoch)
        writer.add_scalar('Spe', spe_val, epoch)
        writer.add_scalar('AUC', auc_val, epoch)

        print(
            f'[{dataset_name}] Epoch {epoch:3d}  '
            f'F1={f1_val:.4f}  Acc={acc_val:.4f}  '
            f'Sen={sen_val:.4f}  Spe={spe_val:.4f}  AUC={auc_val:.4f}'
        )

        # Save best checkpoint (fixed swap bug from original helpfunc.py)
        if f1_val > best_f1:
            best_f1 = f1_val
            best_metrics = {
                'epoch': epoch, 'F1': f1_val, 'Acc': acc_val,
                'Sen': sen_val, 'Spe': spe_val, 'AUC': auc_val
            }
            torch.save(thinmodel.state_dict(),  f'{ckpt_dir}/{dataset_name}_thin.pth')
            torch.save(thickmodel.state_dict(), f'{ckpt_dir}/{dataset_name}_thick.pth')
            torch.save(refine.state_dict(),     f'{ckpt_dir}/{dataset_name}_refine.pth')
            print(f'  *** Best F1={best_f1:.4f} at epoch {epoch} — checkpoint saved ***')

    writer.close()
    return best_metrics


print('Training function defined.')

## 6. Run All Experiments

Trains each selected dataset in sequence. Results are saved to Drive after each run.

In [ ]:
import shutil

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

all_results = {}

for dataset_name, cfg in EXPERIMENTS.items():
    if not RUN_EXPERIMENTS[dataset_name]:
        print(f'\n[SKIP] {dataset_name}')
        continue

    print(f'\n{"="*65}')
    print(f'  Dataset : {dataset_name}')
    print(f'  Epochs  : {cfg["epochs"]}    Batch size: {cfg["batch_size"]}')
    print(f'{"="*65}\n')

    best = train_one_experiment(
        dataset_name = dataset_name,
        train_dir    = cfg['train_dir'],
        test_dir     = cfg['test_dir'],
        epochs       = cfg['epochs'],
        batch_size   = cfg['batch_size'],
        device       = device,
    )
    all_results[dataset_name] = best

    # ── Copy checkpoints + visual outputs + TensorBoard logs to Drive ────────
    drive_out = os.path.join(RESULTS_DRIVE_DIR, dataset_name)
    os.makedirs(drive_out, exist_ok=True)

    for src, dst_name in [
        (f'ckpts/{dataset_name}',                    'ckpts'),
        (f'visual_output/{dataset_name}',            'visual_output'),
    ]:
        if os.path.exists(src):
            shutil.copytree(src, os.path.join(drive_out, dst_name), dirs_exist_ok=True)

    print(f'\n[Done] {dataset_name} results saved to: {drive_out}')
    print(f'  Best metrics: {best}\n')

print('\n' + '='*65)
print('ALL EXPERIMENTS COMPLETE')
print('='*65)

## 7. Results Summary Table

In [ ]:
import pandas as pd

if all_results:
    df = pd.DataFrame(all_results).T
    df.index.name = 'Dataset'
    # Round for readability
    for col in ['F1', 'Acc', 'Sen', 'Spe', 'AUC']:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: round(float(x), 4))
    print(df.to_string())
    # Save as CSV to Drive
    csv_path = os.path.join(RESULTS_DRIVE_DIR, 'results_summary.csv')
    df.to_csv(csv_path)
    print(f'\nSummary saved to: {csv_path}')
else:
    print('No results to display — run the experiments first.')

## 8. Visualize Sample Segmentation Outputs

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob
from pathlib import Path

for dataset_name in EXPERIMENTS:
    if not RUN_EXPERIMENTS.get(dataset_name, False):
        continue

    all_dir = os.path.join(RESULTS_DRIVE_DIR, dataset_name, 'visual_output', 'all')
    imgs    = sorted(glob.glob(os.path.join(all_dir, '*.png')))[:4]

    if not imgs:
        print(f'[{dataset_name}] No output images found at {all_dir}')
        continue

    fig, axes = plt.subplots(1, len(imgs), figsize=(4 * len(imgs), 4))
    if len(imgs) == 1:
        axes = [axes]

    fig.suptitle(f'{dataset_name.upper()} — RefineNet Output (best checkpoint)',
                 fontsize=12, fontweight='bold')
    for ax, p in zip(axes, imgs):
        ax.imshow(mpimg.imread(p), cmap='gray')
        ax.set_title(Path(p).name, fontsize=8)
        ax.axis('off')

    plt.tight_layout()
    out_png = os.path.join(RESULTS_DRIVE_DIR, f'{dataset_name}_preview.png')
    plt.savefig(out_png, dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Saved preview: {out_png}\n')

## 9. (Optional) Launch TensorBoard

View training curves for all four experiments.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ckpts

## 10. (Optional) Generate Thick/Thin Masks from Full Masks

If your dataset only has full vessel masks, run this cell to auto-generate `thick_mask/` and `thin_mask/` folders using morphological open/close operations — same method used in the paper.

In [ ]:
import cv2
import numpy as np

def generate_thick_thin_masks(mask_dir, thick_out_dir, thin_out_dir):
    """Splits full vessel mask into thick and thin components."""
    os.makedirs(thick_out_dir, exist_ok=True)
    os.makedirs(thin_out_dir, exist_ok=True)

    mask_files = [f for f in os.listdir(mask_dir)
                  if f.lower().endswith(('.png', '.jpg', '.bmp', '.ppm', '.tif'))]
    print(f'Processing {len(mask_files)} masks from {mask_dir}')

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    for fname in mask_files:
        img = cv2.imread(os.path.join(mask_dir, fname), 0)
        if img is None:
            print(f'  Could not read {fname}, skipping.')
            continue
        closed       = cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel)
        opened       = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel)
        thick_vessel = cv2.bitwise_and(img, opened)
        thin_vessel  = cv2.bitwise_and(img, cv2.bitwise_not(opened))

        stem = os.path.splitext(fname)[0]
        cv2.imwrite(os.path.join(thick_out_dir, f'{stem}.png'), thick_vessel)
        cv2.imwrite(os.path.join(thin_out_dir,  f'{stem}.png'), thin_vessel)

    print('Done.')


# Example: generate for chasedb1 train split
# generate_thick_thin_masks(
#     mask_dir      = f'{DRIVE_ROOT}/chasedb1/train/mask',
#     thick_out_dir = f'{DRIVE_ROOT}/chasedb1/train/thick_mask',
#     thin_out_dir  = f'{DRIVE_ROOT}/chasedb1/train/thin_mask',
# )

# Uncomment and repeat for each dataset/split as needed
print('Mask generation function loaded. Uncomment the call above to use it.')